### Modelling

In [24]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, StandardScaler
from packaging.version import parse
import sklearn



df_final = pd.read_csv('data_bersih_tmdb.csv')
RS = 42
num_cols   = ['log_budget', 'runtime', 'release_year', 'log_director_prev_films', 'month_sin', 'month_cos']
flag_cols  = ['director_known', 'actor_known']
dummy_cols = [c for c in df_final.columns if c.startswith(('genre_grp_', 'language_grp_', 'country_grp_'))]
te_cols    = ['primary_director', 'lead_actor']

X = df_final[num_cols + flag_cols + dummy_cols + te_cols]
y = df_final['log_revenue']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RS)

if parse(sklearn.__version__) >= parse('1.9'):
    te_kw = dict(cv=KFold(5, shuffle=True, random_state=RS))
else:
    te_kw = dict(cv=5, random_state=RS)

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('te', Pipeline([('enc', TargetEncoder(target_type='continuous', smooth='auto', **te_kw)),
                     ('sc', StandardScaler())]), te_cols),
], remainder='passthrough', verbose_feature_names_out=False)   # flag + dummy dilewatkan apa adanya
pre.set_output(transform='pandas')

ren = {'primary_director': 'director_te', 'lead_actor': 'actor_te'}
X_train_enc = pre.fit_transform(X_train, y_train).rename(columns=ren)
X_test_enc  = pre.transform(X_test).rename(columns=ren)

print(X_train_enc.shape, X_test_enc.shape)
print("Kolom non-numerik:", X_train_enc.select_dtypes(exclude='number').columns.tolist())  # harus []
print("Missing di test:", X_test_enc.isnull().sum().sum())

(10320, 48) (2581, 48)
Kolom non-numerik: []
Missing di test: 0


### Feature selection

#### 5.1 variansi rendah dan multikolinearitas

In [25]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, f_regression, RFECV
from sklearn.linear_model import LassoCV, Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

binary_cols = flag_cols + dummy_cols
nonbin_cols = [c for c in X_train_enc.columns if c not in binary_cols]
fs = pd.DataFrame(index=X_train_enc.columns)

# Dummy dengan prevalensi < 1% dianggap hampir konstan
vt = VarianceThreshold(threshold=0.01 * 0.99).fit(X_train_enc)
fs['low_variance'] = ~vt.get_support()

# Korelasi antar fitur numerik
corr = X_train_enc[nonbin_cols].corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
print("Pasangan korelasi > 0.85:\n", pairs[pairs > 0.85])

# VIF (dihitung manual, tanpa statsmodels). VIF > 5-10 berarti multikolinearitas tinggi
def hitung_vif(d):
    out = {}
    for c in d.columns:
        oth = d.drop(columns=c)
        r2 = LinearRegression().fit(oth, d[c]).score(oth, d[c])
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out).sort_values(ascending=False)
print(hitung_vif(X_train_enc[nonbin_cols]))

Pasangan korelasi > 0.85:
 Series([], dtype: float64)
log_budget                 1.565651
director_te                1.323894
actor_te                   1.253919
release_year               1.206529
log_director_prev_films    1.176451
runtime                    1.062197
month_cos                  1.002878
month_sin                  1.001768
dtype: float64


#### Metode Filter

In [26]:
fs['mi'] = mutual_info_regression(
    X_train_enc, y_train,
    discrete_features=X_train_enc.columns.isin(binary_cols), random_state=RS)
fs['f_pvalue'] = f_regression(X_train_enc, y_train)[1]

#### Metode embedded (Lasso, Random Forest, permutation importance)

In [27]:
import gc

# Pakai float32 supaya memori lebih hemat
X_tr32 = X_train_enc.astype('float32')

# --- Lasso (ringan, tanpa paralel) ---
Xs = StandardScaler().fit_transform(X_tr32)
lasso = LassoCV(cv=3, alphas=50, random_state=RS, n_jobs=1).fit(Xs, y_train)
fs['lasso_coef'] = lasso.coef_
del Xs, lasso; gc.collect()

# --- Random Forest versi ringan ---
X_in, X_val, y_in, y_val = train_test_split(X_tr32, y_train, test_size=0.2, random_state=RS)

# Batasi ukuran validasi untuk permutation importance
if len(X_val) > 3000:
    idx = X_val.sample(3000, random_state=RS).index
    X_val, y_val = X_val.loc[idx], y_val.loc[idx]

rf = RandomForestRegressor(
    n_estimators=100, max_depth=12, min_samples_leaf=5,
    max_features='sqrt', n_jobs=2, random_state=RS
).fit(X_in, y_in)
fs['rf_importance'] = rf.feature_importances_
pm = permutation_importance(rf, X_val, y_val, n_repeats=3, random_state=RS, n_jobs=1)
fs['perm_mean'], fs['perm_std'] = pm.importances_mean, pm.importances_std
del rf, pm, X_in, X_val, y_in, y_val; gc.collect()

1042

#### 5.4 Metode wrapper (RFECV, bisa agak lama)

In [28]:
kf = KFold(3, shuffle=True, random_state=RS)
rfecv = RFECV(
    RandomForestRegressor(n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=RS),
    step=2, cv=kf, scoring='neg_root_mean_squared_error', min_features_to_select=5
).fit(X_train_enc, y_train)
fs['rfecv'] = rfecv.support_
print("Jumlah fitur optimal RFECV:", rfecv.n_features_)

Jumlah fitur optimal RFECV: 46


#### 5.5 Voting dan penentuan fitur akhir

In [33]:
vote = pd.DataFrame({
    'f_test': fs['f_pvalue'] < 0.05,
    'mi':     fs['mi'] > fs['mi'].quantile(0.60),
    'lasso':  fs['lasso_coef'].abs() > 1e-6,
    'perm':   (fs['perm_mean'] > 2 * fs['perm_std']) & (fs['perm_mean'] > 0),
    'rfecv':  fs['rfecv'],
})
fs['votes'] = vote.sum(axis=1)

must_keep = ['log_budget', 'director_te', 'actor_te']
selected = fs.index[((fs['votes'] >= 3) & ~fs['low_variance'])
                    | fs.index.isin(must_keep)].tolist()

print("\n".join(map(str, selected)))

log_budget
runtime
release_year
log_director_prev_films
month_sin
month_cos
director_te
actor_te
genre_grp_Action
genre_grp_Adventure
genre_grp_Animation
genre_grp_Crime
genre_grp_Documentary
genre_grp_Drama
genre_grp_Family
genre_grp_Fantasy
genre_grp_Horror
genre_grp_Other
genre_grp_Romance
genre_grp_Science Fiction
language_grp_Other
language_grp_ar
language_grp_en
language_grp_es
language_grp_fr
language_grp_hi
language_grp_ja
language_grp_ru
language_grp_zh
country_grp_Australia
country_grp_China
country_grp_India
country_grp_Italy
country_grp_Other
country_grp_Russia
country_grp_United States of America


#### 5.6 Validasi: apakah seleksi benar-benar membantu, dan apakah sutradara/pemeran berguna

In [38]:
semua     = X_train_enc.columns.tolist()
kru       = ['director_te', 'actor_te', 'log_director_prev_films',
             'director_known', 'actor_known']
tanpa_kru = [c for c in semua if c not in kru]
inti      = ['log_budget']

def cv_rmse(cols, model):
    s = cross_val_score(model, X_train_enc[cols], y_train, cv=kf,
                        scoring='neg_root_mean_squared_error')
    return -s.mean()

hasil = []
for nama, cols in [('Semua fitur', semua), ('Fitur terpilih', selected),
                   ('Tanpa sutradara/pemeran', tanpa_kru), ('Hanya log_budget', inti)]:
    hasil.append({
        'Skenario': nama, 'n_fitur': len(cols),
        'RMSE Ridge': cv_rmse(cols, Ridge()),
        'RMSE RF': cv_rmse(cols, RandomForestRegressor(
            n_estimators=150, min_samples_leaf=3, n_jobs=-1, random_state=RS)),
    })
hasil_df = pd.DataFrame(hasil).round(4)

X_train_sel = X_train_enc[selected]
X_test_sel  = X_test_enc[selected]

print(hasil_df)
print(X_train_sel.shape, X_test_sel.shape)

                  Skenario  n_fitur  RMSE Ridge  RMSE RF
0              Semua fitur       48      0.6978   0.6822
1           Fitur terpilih       36      0.6986   0.6828
2  Tanpa sutradara/pemeran       43      0.7206   0.7073
3         Hanya log_budget        1      0.7501   0.7532
(10320, 36) (2581, 36)


#### Baseline + modelling + evaluasi

In [39]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
import joblib

def evaluasi(nama, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    return {
        'Model': nama,
        'RMSE (log)': np.sqrt(mean_squared_error(yte, p)),
        'MAE (log)':  mean_absolute_error(yte, p),
        'R2':         r2_score(yte, p),
        'MAE ($)':    mean_absolute_error(np.expm1(yte), np.expm1(p)),
    }

modelku = {
    'Baseline (mean)': DummyRegressor(strategy='mean'),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                           n_jobs=-1, random_state=RS),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RS),
}
res = [evaluasi(n, m, X_train_sel, y_train, X_test_sel, y_test)
       for n, m in modelku.items()]
print(pd.DataFrame(res).round(4).sort_values('RMSE (log)'), '\n')

param = {
    'n_estimators':     [400, 600, 800],
    'learning_rate':    [0.01, 0.02, 0.05],
    'max_depth':        [3, 4, 5, 6],
    'subsample':        [0.5, 0.7, 0.85],
    'min_samples_leaf': [3, 5, 10],
}
rs = RandomizedSearchCV(GradientBoostingRegressor(random_state=RS), param, n_iter=25,
                        cv=kf, scoring='neg_root_mean_squared_error',
                        n_jobs=-1, random_state=RS).fit(X_train_sel, y_train)
print(rs.best_params_, 'CV RMSE:', round(-rs.best_score_, 4))

# Evaluasi model final di test set
final = rs.best_estimator_
p = final.predict(X_test_sel)
print({
    'RMSE (log)': round(np.sqrt(mean_squared_error(y_test, p)), 4),
    'MAE (log)':  round(mean_absolute_error(y_test, p), 4),
    'R2':         round(r2_score(y_test, p), 4),
    'MAE ($)':    round(mean_absolute_error(np.expm1(y_test), np.expm1(p)), 2),
    'MedAE ($)':  round(np.median(np.abs(np.expm1(y_test) - np.expm1(p))), 2),
})

joblib.dump({'pre': pre, 'rename': ren, 'selected': selected, 'model': final},
            'model_tmdb.joblib')

               Model  RMSE (log)  MAE (log)      R2    MAE ($)
2      Random Forest      0.6651     0.4958  0.6119   614.0701
3  Gradient Boosting      0.6652     0.5016  0.6118   624.5312
1              Ridge      0.6863     0.5185  0.5868   661.1937
0    Baseline (mean)      1.0680     0.8607 -0.0008  1096.0097 

{'subsample': 0.5, 'n_estimators': 600, 'min_samples_leaf': 5, 'max_depth': 6, 'learning_rate': 0.01} CV RMSE: 0.6641
{'RMSE (log)': np.float64(0.6525), 'MAE (log)': 0.4866, 'R2': 0.6264, 'MAE ($)': 602.81, 'MedAE ($)': np.float64(374.89)}


['model_tmdb.joblib']